# Course 3: Large Language Models
## Session 9: Multimodal models

**Goal:** train a miniature CLIP with the contrastive loss, use it for zero-shot classification, and plan a probe of a vision-language model on Bangla text in images.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

### 1. A miniature CLIP
Two encoders, one for images and one for text, trained so that an image and its caption land close together and mismatched pairs land far apart. Images are the 8x8 digits from scikit-learn. Captions are short Bangla phrases.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
torch.manual_seed(0); rng = np.random.default_rng(0)
d = load_digits(); X = torch.tensor(d.data / 16.0, dtype=torch.float32); y = torch.tensor(d.target)
perm = torch.randperm(len(X)); tr, te = perm[:1300], perm[1300:]

NAMES = ["শূন্য", "এক", "দুই", "তিন", "চার", "পাঁচ", "ছয়", "সাত", "আট", "নয়"]
TEMPLATES = ["এটি {} সংখ্যার ছবি", "হাতে লেখা {}", "{} এর একটি ছবি", "সংখ্যা {}"]
words = sorted({w for t in TEMPLATES for n in NAMES for w in t.format(n).split()}); W = {w: i for i, w in enumerate(words)}
def caption_ids(text): return torch.tensor([W[w] for w in text.split()])

class TextEncoder(nn.Module):
    def __init__(self, d=32): super().__init__(); self.emb = nn.Embedding(len(words), 64); self.out = nn.Linear(64, d)
    def forward(self, captions): return self.out(torch.stack([self.emb(caption_ids(c)).mean(0) for c in captions]))
image_enc = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 32)); text_enc = TextEncoder()
logit_scale = nn.Parameter(torch.tensor(2.0))

### 2. The contrastive loss
In a batch of n pairs, build the n x n table of similarities. Each image should pick its own caption out of the row, and each caption its own image out of the column.

In [ ]:
def clip_loss(img, txt):
    img, txt = F.normalize(img, dim=-1), F.normalize(txt, dim=-1)
    logits = logit_scale.exp() * img @ txt.T
    target = torch.arange(len(img))
    return (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2

opt = torch.optim.Adam([*image_enc.parameters(), *text_enc.parameters(), logit_scale], lr=2e-3)
for step in range(400):
    cls = torch.randperm(10)[:8]                                           # 8 different digits, so no false negatives in the batch
    idx = torch.stack([tr[(y[tr] == c).nonzero().squeeze(1)][torch.randint((y[tr] == c).sum().item(), (1,))] for c in cls]).squeeze(1)
    caps = [TEMPLATES[rng.integers(3)].format(NAMES[c]) for c in cls.tolist()]     # the fourth template is never used in training
    loss = clip_loss(image_enc(X[idx]), text_enc(caps)); opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0 or step == 399: print(f"step {step:3d}  loss {loss.item():.3f}")

### 3. Zero-shot classification
No classifier was trained. To classify an image, embed one caption per class and choose the closest.

In [ ]:
@torch.no_grad()
def zero_shot(template):
    T_ = F.normalize(text_enc([template.format(n) for n in NAMES]), dim=-1)
    return ((F.normalize(image_enc(X[te]), dim=-1) @ T_.T).argmax(1) == y[te]).float().mean().item()
for t in TEMPLATES: print(f"prompt '{t}': accuracy {zero_shot(t):.3f}" + ("   <- template never seen in training" if t == TEMPLATES[3] else ""))
assert zero_shot(TEMPLATES[0]) > 0.85

The wording of the prompt changes the accuracy. That holds for the real CLIP too, which is why papers average over many prompt templates.

### 4. Retrieval in the other direction: text to image

In [ ]:
with torch.no_grad():
    q = F.normalize(text_enc(["হাতে লেখা সাত"]), dim=-1); sims = (F.normalize(image_enc(X[te]), dim=-1) @ q.T).squeeze(1)
top = sims.topk(8).indices
print("labels of the 8 images closest to 'হাতে লেখা সাত':", y[te][top].tolist())

### 5. Real CLIP, and Bangla text inside images (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install transformers pillow`, model downloads, and a Bangla font file for rendering text.
from PIL import Image, ImageDraw, ImageFont
from transformers import CLIPModel, CLIPProcessor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32"); proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Part A: zero-shot classification with English and with Bangla class names. Expect Bangla names to do worse: why?
# inputs = proc(text=["a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)
# probs = model(**inputs).logits_per_image.softmax(-1)

# Part B: render Bangla words into images, then ask a vision-language model to read them.
def render(text, font_path="NotoSansBengali-Regular.ttf", size=48):
    img = Image.new("RGB", (640, 120), "white")
    ImageDraw.Draw(img).text((20, 30), text, font=ImageFont.truetype(font_path, size), fill="black")   # check conjuncts render correctly
    return img
# Test set: 50 images covering (a) simple words, (b) words with conjuncts such as ক্ষ and জ্ঞ, (c) numerals, (d) a shop sign photo.
# For each, ask the model 'What is written in this image?' and score exact match and character error rate.

### 6. Lip reading as a multimodal problem
Audio-visual speech recognition (for example AV-HuBERT) learns from video of the mouth together with audio, and can then transcribe from the video alone. For Bangla, the open questions are data (hours of aligned video) and visemes: which Bangla sounds look identical on the lips?

### TODO
1. Hold one digit out of training entirely. Can the model classify it zero-shot? Why not, and what would it need?
2. Train with batches that contain repeated classes (false negatives). How much does accuracy drop?
3. Run Part B and report the character error rate for each of the four groups.